# 3D Gaussian Splatting (Nerfstudio) 実行ノートブック
Colab上で動画から3Dモデル(.ply)を生成し、ダウンロードするまでの一連の手順です。

**※重要※**
実行前に、メニューの「ランタイム」>「ランタイムのタイプを変更」から、ハードウェア アクセラレータを「**T4 GPU**」に設定してください。

In [ ]:
# @title 1. 環境構築 (Nerfstudio等のインストール)
# 数分かかります。
!pip install nerfstudio
!apt-get update && apt-get install -y colmap ffmpeg

In [ ]:
# @title 2. 動画ファイルのアップロード
# セルを実行するとファイル選択ボタンが表示されます。
# 30秒程度の動画ファイル (.mp4, .mov等) をアップロードしてください。
from google.colab import files
import os
import shutil

upload_dir = '/content/video_data'
os.makedirs(upload_dir, exist_ok=True)

print("動画ファイルを選択してアップロードしてください。")
uploaded = files.upload()

video_path = ""
for filename in uploaded.keys():
    new_path = os.path.join(upload_dir, filename)
    shutil.move(filename, new_path)
    video_path = new_path
    print(f"アップロード完了: {video_path}")

In [ ]:
# @title 3. 動画から学習データの切り出し (COLMAP)
# 勉強会の資料に基づき、150フレーム目標で切り出します。CPUで実行します。
output_dir = '/content/processed_data'

# 前回実行時のデータが残っていれば削除
!rm -rf {output_dir}

!ns-process-data video --data "{video_path}" --output-dir "{output_dir}" --num-frames-target 150 --no-gpu

In [ ]:
# @title 4. 3D Gaussian Splattingの学習
# GPUメモリ不足回避のためViewerは起動せず、ログはTensorBoardに出力します。学習完了後に自動で終了します。
!ns-train splatfacto --data "{output_dir}" --vis tensorboard

In [ ]:
# @title 5. 3Dモデル(.ply)の書き出しとダウンロード
import glob
import os
from google.colab import files

# 学習結果のconfig.ymlを検索
config_paths = glob.glob('/content/outputs/processed_data/splatfacto/*/config.yml')

if not config_paths:
    print("学習結果が見つかりません。")
else:
    # 最新のconfigを使用
    config_paths.sort(key=os.path.getmtime, reverse=True)
    config_path = config_paths[0]
    
    export_dir = '/content/exports'
    os.makedirs(export_dir, exist_ok=True)
    
    print("3Dモデル(.ply)を生成しています...")
    !ns-export gaussian-splat --load-config "{config_path}" --output-dir "{export_dir}"
    
    ply_files = glob.glob(os.path.join(export_dir, '*.ply'))
    if ply_files:
        # 最新のplyファイルをダウンロード
        ply_files.sort(key=os.path.getmtime, reverse=True)
        ply_file = ply_files[0]
        print(f"ダウンロードを開始します: {ply_file}")
        files.download(ply_file)
    else:
        print("plyファイルの生成に失敗しました。")